# AML - MARK LIST FILES - CROSS-REFERENCE 
This script cross-references variants in the SAIGE-GENE+ marker list with somatic variants from the GDC and germline variants from ClinVar. Analyses are conducted separately for ultra-rare variants (MAC ≤ 10) and rare variants (MAF < 1% and MAC > 10). AoU IDs are converted to HGVS IDs, and the GDC somatic mutation file is annotated to indicate the presence or absence of each variant. For ClinVar, variants are filtered to retain only those present in the marker list, resulting in a final consolidated list of variants.


In [ ]:
# Load packages 
import os 
import pandas as pd 
import glob


In [ ]:
# Set environment variable for workspace bucket 
my_bucket = os.getenv('WORKSPACE_BUCKET')

## DOWNLOAD RESULTS FROM THE WORKSPACE BUCKET 

In [ ]:
# Delete directories for marker lists
!rm -rf singleAssoc
!rm -rf markerList
!rm -rf annotationGroup

In [ ]:
# Create directories for marker lists
!mkdir -p singleAssoc
!mkdir -p markerList
!mkdir -p annotationGroup

In [ ]:
%%bash 
# Download single variant association and marker files in loop 
for chrom in {1..22}; do 
    echo "Processing chromosome ${chrom} ...."
    gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr${chrom}.singleAssoc.txt singleAssoc
    gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr${chrom}.markerList.txt markerList
    
done 
# Download annotation files 
gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/annotation_table/EUR_HC_annotation/* annotationGroup

In [ ]:
# Load and concatenate all single variant association files
files = glob.glob('singleAssoc/*.txt')
singleAssoc = [pd.read_csv(file, sep ='\t') for file in files]
singleAssoc = pd.concat(singleAssoc, axis = 0)

In [ ]:
# Load and concatenate all marker list files
files = glob.glob('markerList/*.txt')
markerList = [pd.read_csv(file, sep ='\t') for file in files]
markerList = pd.concat(markerList, axis = 0)

In [ ]:
# Filter maker list for genes of interest 
# Provide a list of genes which were genome-wide significant 
genes_of_interest = [
    "ENSG00000119772",
    "ENSG00000168769",
    "ENSG00000161547",
    "ENSG00000182054"
]
markerList = (
    markerList[markerList['Region']
               .isin(genes_of_interest)]
)
# Provide a list of annotation groups of interest 
group_of_interest = [
    "LOFTEE-HC;SpliceAI-0.8",
    "AlphaMissense-0.564",
    "LOFTEE-HC;SpliceAI-0.8;AlphaMissense-0.564",
]
markerList = (
    markerList[markerList['Group']
               .isin(group_of_interest)]
)


## ULTRA-RARE MARKER LIST 

In [ ]:
# Format ultra-rare marker list 
Ultra_Rare_Variants = (
    markerList[["Region", "Group", "Ultra_Rare_Variants"]]
)
Ultra_Rare_Variants['Ultra_Rare_Variants'] = (
    Ultra_Rare_Variants['Ultra_Rare_Variants'].str.split(',')
)
Ultra_Rare_Variants = (
    Ultra_Rare_Variants.explode('Ultra_Rare_Variants')
    .reset_index(drop=True)
)
Ultra_Rare_Variants = (
    Ultra_Rare_Variants[~Ultra_Rare_Variants["Ultra_Rare_Variants"].isna()]
)

In [ ]:
# EXTRA CODE (TO BE DELETED?)
UltraRareProteinPaint = Ultra_Rare_Variants 
UltraRareProteinPaint = UltraRareProteinPaint.rename(columns = {"Ultra_Rare_Variants":"Variant"})
split_variant_id = UltraRareProteinPaint['Variant'].str.split(':')
UltraRareProteinPaint['Position'] = split_variant_id.str[1]
UltraRareProteinPaint['Class'] = 'WT'
UltraRareProteinPaint = UltraRareProteinPaint[['Region', 'Variant', 'Position', 'Class']]

In [ ]:
# Fitler ultra-rare marker list for loss-of-function and missense variants 
Ultra_Rare_Variants = (
Ultra_Rare_Variants[(Ultra_Rare_Variants["Group"] == "AlphaMissense-0.564") | 
                    (Ultra_Rare_Variants["Group"] == "LOFTEE-HC;SpliceAI-0.8")]
)

Ultra_Rare_Variants = (
    Ultra_Rare_Variants.
    drop_duplicates(subset=["Ultra_Rare_Variants"])
)
# Save marker list of ultra-rare variants 
Ultra_Rare_Variants.to_csv("EUR-AML-Ultra-Rare-Variants.markerList.txt", 
                           sep='\t', index = None)

## RARE MARKER LIST 

In [ ]:
# Format rare marker list 
Rare_Variants = (
    markerList[["Region", "Group", "Rare_Variants"]]
)
Rare_Variants['Rare_Variants'] = (
    Rare_Variants['Rare_Variants'].str.split(',')
)
Rare_Variants = (
    Rare_Variants.explode('Rare_Variants')
    .reset_index(drop=True)
)
Rare_Variants = (
    Rare_Variants[~Rare_Variants["Rare_Variants"].isna()]
)

In [ ]:
# EXTRA CODE (TO BE DELETED?)
RareProteinPaint = Rare_Variants 
RareProteinPaint = RareProteinPaint.rename(columns = {"Rare_Variants":"Variant"})
split_variant_id = RareProteinPaint['Variant'].str.split(':')
RareProteinPaint['Position'] = split_variant_id.str[1]
RareProteinPaint['Class'] = 'WT'
RareProteinPaint = RareProteinPaint[['Region', 'Variant', 'Position', 'Class']]

In [ ]:
# Fitler rare marker list for loss-of-function and missense variants 
Rare_Variants = (
Rare_Variants[(Rare_Variants["Group"] == "AlphaMissense-0.564") | 
              (Rare_Variants["Group"] == "LOFTEE-HC;SpliceAI-0.8")]
)

Rare_Variants = (
    Rare_Variants
    .drop_duplicates(subset=["Rare_Variants"])
)
# Save marker list of rare variants 
Rare_Variants.to_csv("EUR-AML-Rare-Variants.markerList.txt", 
                     sep='\t', index = None)

In [ ]:
# Filter single variant association file for variants in marker list 
singleAssoc = (
    singleAssoc[singleAssoc["MarkerID"]
                .isin(Rare_Variants["Rare_Variants"].unique())]
)
# Save single variant association file 
singleAssoc.to_csv("EUR-AML-Rare-Variants.singleAssoc.txt", 
                   sep='\t', index = None)

## SAVE RESULTS 

In [ ]:
# Copy marker list results to workspace bucket 
!gsutil cp EUR-AML-Rare-Variants.markerList.txt {my_bucket}/data/saige_step02/HC_marklist/
!gsutil cp EUR-AML-Ultra-Rare-Variants.markerList.txt {my_bucket}/data/saige_step02/HC_marklist/
!gsutil cp EUR-AML-Rare-Variants.singleAssoc.txt {my_bucket}/data/saige_step02/HC_marklist/

## GDC CROSS-REFERENCE  

In [ ]:
# Download marker lists to markerlist folder 
!gsutil cp {my_bucket}/data/saige_step02/HC_marklist/* markerList

In [ ]:
# Split and format AoU IDs into chromosome, position, reference, and alternate alleles for ultra-rare variants
Ultra_Rare_Variants[['CHR', 'POS', 'REF', 'ALT']] = (
    Ultra_Rare_Variants['Ultra_Rare_Variants']
    .str.split(":", expand=True)
)
Ultra_Rare_Variants['POS'] = (
    Ultra_Rare_Variants['POS'].astype(int)
)

# Split and format AoU IDs into chromosome, position, reference, and alternate alleles for rare variants
Rare_Variants[['CHR', 'POS', 'REF', 'ALT']] = (
    Rare_Variants['Rare_Variants']
    .str.split(":", expand=True)
)
Rare_Variants['POS'] = (
    Rare_Variants['POS'].astype(int)
)

In [ ]:
# ==========================================================================================================
# Convert AoU IDs to HGVS IDs 
# ==========================================================================================================

# Function to convert AoU IDs to HGVS IDs 
def VCF_to_HGVS(row):
    pos = int(row["POS"])
    ref = row["REF"]
    alt = row["ALT"]
    chr = row["CHR"]
    # Substitution 
    if len(ref) == len(alt):
        return f"{chr}:g.{pos}{ref}>{alt}"
    # Deletion 
    elif len(ref) > len(alt):
        return f"{chr}:g.{pos + 1}del{ref[1:]}"
    # Insertion 
    elif len(ref) < len(alt):
        end_pos = pos + len(alt[1:])
        return f"{chr}:g.{pos}_{end_pos}ins{alt[1:]}"

# Apply to ultra-rare marker list 
Ultra_Rare_Variants["HGVS"] = Ultra_Rare_Variants.apply(VCF_to_HGVS, axis=1)

# Apply to rare marker list 
Rare_Variants["HGVS"] = Rare_Variants.apply(VCF_to_HGVS, axis=1)


In [ ]:
# Save ultra-rare HGVS IDs file 
Ultra_Rare_Variants.to_csv("EUR-AML-Ultra-Rare-HGVS.markerList.txt", 
                           sep='\t', index = None)
# Save rare HGVS IDs file 
Rare_Variants.to_csv("EUR-AML-Rare-HGVS.markerList.txt", 
                           sep='\t', index = None)

In [ ]:
# Copy HGVS IDs files to workspace bucket 
!gsutil cp EUR-AML-Ultra-Rare-HGVS.markerList.txt {my_bucket}/data/saige_step02/HC_marklist/
!gsutil cp EUR-AML-Rare-HGVS.markerList.txt {my_bucket}/data/saige_step02/HC_marklist/

In [ ]:
# Load and concatenate GDC somatic variant files 
files = glob.glob('markerList/*.tsv')
frequent_mutations = [pd.read_csv(file, sep ='\t') for file in files]
frequent_mutations = pd.concat(frequent_mutations, axis = 0)

In [ ]:
# Check if ultra-rare variants are present in GDC somatic variant list 
frequent_mutations["AoU_Ultra_Rare_Variants"] = frequent_mutations["dna_change"].apply(
    lambda x: "Match" if x in Ultra_Rare_Variants["HGVS"].values else "No Match"
)
# Check if ultra-rare variants are present in GDC somatic variant list 
frequent_mutations["AoU_Rare_Variants"] = frequent_mutations["dna_change"].apply(
    lambda x: "Match" if x in Rare_Variants["HGVS"].values else "No Match"
)
# Save GDC somatic cross reference file
frequent_mutations.to_csv("GDC-freq# Copy GDC somatic cross reference file to workspace bucket uent-mutations.tsv", sep='\t', index = None)

In [ ]:
# Copy GDC somatic cross reference file to workspace bucket 
!gsutil cp GDC-frequent-mutations.tsv {my_bucket}/data/saige_step02/HC_marklist/

## CLINVAR CROSS-REFERENCE  

In [ ]:
# Download Clinvar file 
!wget https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz 
# Fitler Clinvar file     
!zgrep  -v "##" clinvar.vcf.gz > clinvar.vcf

In [ ]:
# ==========================================================================================================
# Loop for Clinvar cross reference 
# ==========================================================================================================

# Load Clinvar file 
file_path = "clinvar.vcf"

chunk_size = 10000
vcf_iterator = pd.read_csv(
    file_path,
    sep="\t",          
    chunksize=chunk_size
)

# Create list for cross referenced variants 
rare_clinvar = []
ultra_rare_clinvar = []

# Loop through variant chunks in Clinvar file 
for i, chunk in enumerate(vcf_iterator):
    print(f"Processing chunk {i+1}")
    
    # Format variant IDs to match AoU IDs 
    for col in ["#CHROM", "POS", "REF", "ALT"]:
        chunk[col] = chunk[col].astype(str)
        
    chunk["Variants"] = "chr" + chunk["#CHROM"] + ":" + chunk["POS"] + ":" + chunk["REF"] + ":" + chunk["ALT"]
    # Check if AoU variant is present within Clinvar 
    ultra_rare_clinvar.append(chunk[chunk["Variants"].isin(Ultra_Rare_Variants["Ultra_Rare_Variants"])])
    rare_clinvar.append(chunk[chunk["Variants"].isin(Rare_Variants["Rare_Variants"])])
    
ultra_rare_clinvar = pd.concat(ultra_rare_clinvar, axis = 0)
rare_clinvar = pd.concat(rare_clinvar, axis = 0)

In [ ]:
# Extract CLNDN using regex for ultra-rare variants 
ultra_rare_clinvar["CLNDN"] = (
    ultra_rare_clinvar["INFO"].str.extract(r"CLNDN=([^;]+)")
)
# Extract CLNDN using regex for rare variants 
rare_clinvar["CLNDN"] = (
    rare_clinvar["INFO"].str.extract(r"CLNDN=([^;]+)")
)
# Save ultra-rare Clinvar variants file 
ultra_rare_clinvar.to_csv("EUR-AML-Ultra-Rare-Clinvar.markerList.txt", 
                          sep='\t', index = None)
# Save rare Clinvar variants file 
rare_clinvar.to_csv("EUR-AML-Rare-Clinvar.markerList.txt", 
                    sep='\t', index = None)

In [ ]:
# Copy ultra-rare Clinvar variants file to workspace bucket 
!gsutil cp EUR-AML-Ultra-Rare-Clinvar.markerList.txt {my_bucket}/data/saige_step02/HC_marklist/
# Copy Grare Clinvar variants file to workspace bucket 
!gsutil cp EUR-AML-Rare-Clinvar.markerList.txt {my_bucket}/data/saige_step02/HC_marklist/

# ANNOTATION GROUP FILE 

In [ ]:
# Load and concatenate annotation files 
files = glob.glob('annotationGroup/*.txt')
annotation_group = [pd.read_csv(file, sep ='\t', header=None) for file in files]
annotation_group = pd.concat(annotation_group, axis = 0)

In [ ]:
# Filter annotation file for genes of interest 
annotation_group = (
    annotation_group[
        annotation_group[0].str.contains("ENSG00000119772|ENSG00000168769|ENSG00000161547|ENSG00000182054")]
)

annotation_group = annotation_group[0].str.split(" ", expand=True)

# Format fitlered annotation file 
annotation_group = (
    annotation_group
    .set_index([0, 1])  
    .stack()                   
    .unstack(1)             
    .reset_index()         
    .rename(columns={0: 'Region'})
    .drop(columns='level_1')
)

# Save filtered annoation file 
annotation_group.to_csv("Group-Annotation.txt", sep ='\t', index=None)

In [ ]:
# Copy filtered annotation file to workspace bucket 
!gsutil cp Group-Annotation.txt {my_bucket}/data/saige_step02/HC_marklist/ 

In [ ]:
# EXTRA CODE (TO BE DELETED?)
ProteinPaint = pd.concat([RareProteinPaint, UltraRareProteinPaint]) 

for gene in ProteinPaint['Region'].unique():
    print(f"Processing gene {gene} ....")
    ProteinPaintGene = ProteinPaint[ProteinPaint['Region'] == gene]
    ProteinPaintGene = ProteinPaintGene.drop(columns = "Region")
    ProteinPaintGene.to_csv(f"protein_paint_{gene}.tsv", sep =',', 
                            index = None, header=None)